# LeetCode 3530 — Maximum Profit from Valid Topological Order in DAG bằng Python

- Link: https://leetcode.com/problems/maximum-profit-from-valid-topological-order-in-dag/
- Độ khó: **Hard**
- Chủ đề: **Bitmask DP trên tập đỉnh đã xử lý** (ideal của DAG), **thứ tự topo**, bất đẳng thức sắp xếp lại, tăng tốc bằng **vector hóa numpy theo lớp popcount**

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def maxProfit(self, n: int, edges: List[List[int]], score: List[int]) -> int:
```

## 1. Mục tiêu học

1. Hiểu lợi nhuận `Σ score[v] · vị_trí(v)` và vì sao không có tham lam đơn giản khi có ràng buộc thứ tự (bài lập lịch có ràng buộc tiền nhiệm là NP-khó tổng quát).
2. Nhận ra: nếu đã xử lý tập `mask` thì đỉnh tiếp theo nhận vị trí `|mask| + 1` — **chỉ phụ thuộc kích thước tập** ⇒ DP `dp[mask]`.
3. Biểu diễn tiền nhiệm bằng bitmask `pred[v]` và kiểm tra "đã đủ tiền nhiệm" bằng `pred[v] & mask == pred[v]`.
4. Dùng **bất đẳng thức sắp xếp lại** cho trường hợp không có cạnh (kiểm thử nhanh).
5. Tăng tốc bằng numpy: xử lý cả **lớp** các mask cùng popcount một lúc.

## 2. Đề bài & ví dụ

Cho DAG `n` đỉnh, cạnh `u → v` nghĩa là `u` phải đứng trước `v`. Đỉnh `i` có điểm `score[i]`. Xử lý các đỉnh theo một **thứ tự topo**; đỉnh ở vị trí thứ `p` (tính từ 1) đóng góp `score · p`. Trả về tổng lớn nhất.

**Ví dụ 1:** `n=2, edges=[[0,1]], score=[2,3]` → `8` (`0` rồi `1`: `2·1 + 3·2`).

**Ví dụ 2:** `n=3, edges=[[0,1],[0,2]], score=[1,6,3]` → `25` (`0, 2, 1`: `1 + 6 + 18`).

**Ràng buộc:** `1 ≤ n ≤ 22`, `1 ≤ score[i] ≤ 10^5`, đồ thị là DAG, không cạnh lặp.

In [1]:
from typing import List
examples = [(2, [[0, 1]], [2, 3], 8), (3, [[0, 1], [0, 2]], [1, 6, 3], 25)]
for n, e, s, exp in examples:
    print(n, e, s, '-> kỳ vọng', exp)

2 [[0, 1]] [2, 3] -> kỳ vọng 8
3 [[0, 1], [0, 2]] [1, 6, 3] -> kỳ vọng 25


## 3. Kiến thức nền

### 3.1. Thứ tự topo và kiểm tra hợp lệ

Một thứ tự hợp lệ ⇔ với mọi cạnh `u → v`, `u` đứng trước `v`.

In [2]:
def is_topological(order, edges):
    pos = {v: i for i, v in enumerate(order)}
    return all(pos[u] < pos[v] for u, v in edges)

def profit(order, score):
    return sum(score[v] * (i + 1) for i, v in enumerate(order))

print(is_topological([0, 2, 1], [[0, 1], [0, 2]]), profit([0, 2, 1], [1, 6, 3]))

True 25


### 3.2. Bitmask tiền nhiệm

`pred[v]` = bitmask các đỉnh phải đứng trước `v`. Đỉnh `v` "sẵn sàng" khi tập đã xử lý `mask` chứa hết tiền nhiệm: `pred[v] & mask == pred[v]`.

In [3]:
n, edges = 3, [[0, 1], [0, 2]]
pred = [0] * n
for u, v in edges:
    pred[v] |= 1 << u
print('pred =', [bin(p) for p in pred])
mask = 0b001
print('sẵn sàng khi đã xử lý {0}:', [v for v in range(n) if not mask >> v & 1 and pred[v] & mask == pred[v]])

pred = ['0b0', '0b1', '0b1']
sẵn sàng khi đã xử lý {0}: [1, 2]


### 3.3. Bất đẳng thức sắp xếp lại

Không có ràng buộc ⇒ tổng `Σ score · vị_trí` lớn nhất khi xếp **điểm tăng dần** (điểm lớn nhân hệ số lớn).

In [4]:
from itertools import permutations
sc = [5, 1, 3]
best = max(profit(p, sc) for p in permutations(range(3)))
print('vét cạn:', best, '| sắp tăng dần:', sum(s * (i + 1) for i, s in enumerate(sorted(sc))))

vét cạn: 22 | sắp tăng dần: 22


### 3.4. Đếm bit (popcount) cho mọi mask

`pc[m] = pc[m >> 1] + (m & 1)` — tính cho mọi `m < 2^n` trong `O(2^n)`.

In [5]:
pc = [0] * 16
for m in range(1, 16):
    pc[m] = pc[m >> 1] + (m & 1)
print(pc)

[0, 1, 1, 2, 1, 2, 2, 3, 1, 2, 2, 3, 2, 3, 3, 4]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Giả sử đã xử lý xong tập `mask` (theo một thứ tự hợp lệ nào đó, lợi nhuận tốt nhất `dp[mask]`). Đỉnh tiếp theo `v` sẽ ở vị trí `|mask| + 1` **bất kể** thứ tự bên trong `mask`. Vậy:

`dp[mask ∪ {v}] = max(…, dp[mask] + score[v] · (|mask| + 1))` nếu `v` sẵn sàng.

Tính tay ví dụ 2 (`score = [1, 6, 3]`, `0 → 1`, `0 → 2`):

| mask | dp | đỉnh sẵn sàng tiếp |
|---|---|---|
| {} | 0 | 0 |
| {0} | 1·1 = 1 | 1, 2 |
| {0,1} | 1 + 6·2 = 13 | 2 |
| {0,2} | 1 + 3·2 = 7 | 1 |
| {0,1,2} | max(13 + 3·3, 7 + 6·3) = **25** | — |

## 5. Cách 1 — Brute Force: thử mọi hoán vị

In [6]:
def max_profit_brute(n, edges, score):
    best = 0
    for p in permutations(range(n)):
        if is_topological(p, edges):
            best = max(best, profit(p, score))
    return best

for n, e, s, exp in examples:
    print(max_profit_brute(n, e, s), '| kỳ vọng', exp)

8 | kỳ vọng 8
25 | kỳ vọng 25


**Độ phức tạp:** `O(n! · (n + m))` — chỉ dùng với `n ≤ 8`.

**Nút thắt:** lợi nhuận tương lai chỉ phụ thuộc **tập** đã xử lý, không phụ thuộc thứ tự bên trong ⇒ gộp các hoán vị có cùng tiền tố-tập.

## 6. Ý tưởng thuật toán / bước đột phá

`dp[mask]` = lợi nhuận lớn nhất khi các đỉnh trong `mask` chiếm các vị trí `1..|mask|` theo một thứ tự hợp lệ (−∞ nếu `mask` không phải "tập đóng dưới" — có đỉnh mà tiền nhiệm chưa có mặt).

- `dp[0] = 0`.
- Từ `mask`, với mọi `v ∉ mask` sẵn sàng: `dp[mask | 1<<v] = max(…, dp[mask] + score[v] · (popcount(mask) + 1))`.
- Đáp án `dp[full]`.

Duyệt `mask` tăng dần là đủ (mask mới luôn lớn hơn mask cũ).

Vì sao không tham lam? Ví dụ: một đỉnh điểm thấp "mở khóa" một đỉnh điểm rất cao — đôi khi nên xử lý sớm, đôi khi không; bài toán tổng quát tương đương lập lịch có ràng buộc tiền nhiệm (NP-khó), nên cần DP trên tập.

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Đệ quy có nhớ theo `mask`

In [7]:
from functools import lru_cache

def max_profit_memo(n, edges, score):
    pred = [0] * n
    for u, v in edges:
        pred[v] |= 1 << u
    full = (1 << n) - 1

    @lru_cache(maxsize=None)
    def best(mask):                 # lợi nhuận tốt nhất cho phần CÒN LẠI khi đã xử lý mask
        if mask == full:
            return 0
        pos = bin(mask).count('1') + 1
        res = 0
        for v in range(n):
            if not mask >> v & 1 and pred[v] & mask == pred[v]:
                res = max(res, score[v] * pos + best(mask | 1 << v))
        return res

    return best(0)

for n, e, s, exp in examples:
    print(max_profit_memo(n, e, s), '| kỳ vọng', exp)

8 | kỳ vọng 8
25 | kỳ vọng 25


Độ sâu đệ quy chỉ `n ≤ 22`, nhưng số trạng thái tới `2^22 ≈ 4·10^6` — tốn bộ nhớ `lru_cache` và thời gian gọi hàm.

### 7.2. Cách 3 — Bottom-up trên mảng `dp[2^n]` (lời giải nộp bài, Python thuần)

In [8]:
def max_profit_dp(n, edges, score):
    pred = [0] * n
    for u, v in edges:
        pred[v] |= 1 << u
    full = (1 << n) - 1
    dp = [-1] * (1 << n)
    dp[0] = 0
    pc = [0] * (1 << n)
    for m in range(1, 1 << n):
        pc[m] = pc[m >> 1] + (m & 1)
    for mask in range(1 << n):
        d = dp[mask]
        if d < 0:
            continue                       # tập không hợp lệ (không đạt được)
        c = pc[mask] + 1
        rest = full ^ mask
        while rest:
            lb = rest & -rest
            rest ^= lb
            v = lb.bit_length() - 1
            if pred[v] & mask == pred[v]:
                val = d + score[v] * c
                if val > dp[mask | lb]:
                    dp[mask | lb] = val
    return dp[full]

for n, e, s, exp in examples:
    print(max_profit_dp(n, e, s), '| kỳ vọng', exp)

8 | kỳ vọng 8
25 | kỳ vọng 25


**Độ phức tạp:** `O(2^n · n)` thời gian, `O(2^n)` bộ nhớ. Với `n = 22` khoảng `4·10^6 · 11 ≈ 5·10^7` bước trong trường hợp xấu (đồ thị gần như không có cạnh) — vài giây đến hơn chục giây trong Python thuần (xem 12.3).

### 7.3. Cách 4 — Vector hóa bằng numpy theo lớp popcount

Mọi mask cùng popcount `c` chỉ phụ thuộc lớp `c − 1`. Xử lý cả lớp bằng mảng numpy: với mỗi đỉnh `v`, lọc các mask trong lớp chưa chứa `v` và đã đủ tiền nhiệm, cập nhật `dp[mask | 1<<v]` bằng `np.maximum`. Khối lượng `O(n · 2^n)` phép toán mảng (chạy trong C).

In [9]:
import numpy as np

def max_profit_numpy(n, edges, score):
    pred = [0] * n
    for u, v in edges:
        pred[v] |= 1 << u
    N = 1 << n
    masks = np.arange(N, dtype=np.int64)
    pc = np.zeros(N, dtype=np.int64)
    for b in range(n):
        pc += (masks >> b) & 1
    order = np.argsort(pc, kind='stable')
    bounds = np.searchsorted(pc[order], np.arange(n + 2))
    dp = np.full(N, -1, dtype=np.int64)
    dp[0] = 0
    for c in range(n):
        layer = order[bounds[c]:bounds[c + 1]]
        layer = layer[dp[layer] >= 0]          # chỉ các tập hợp lệ
        if layer.size == 0:
            continue
        base = dp[layer]
        for v in range(n):
            sel = ((layer >> v) & 1) == 0
            if pred[v]:
                sel &= (layer & pred[v]) == pred[v]
            src = layer[sel]
            if src.size == 0:
                continue
            tgt = src | (1 << v)                # các tgt khác nhau với v cố định
            dp[tgt] = np.maximum(dp[tgt], base[sel] + score[v] * (c + 1))
    return int(dp[N - 1])

for n, e, s, exp in examples:
    print(max_profit_numpy(n, e, s), '| kỳ vọng', exp)

8 | kỳ vọng 8
25 | kỳ vọng 25


Với một `v` cố định, các `tgt = src | (1<<v)` đôi một khác nhau nên phép gán `dp[tgt] = np.maximum(...)` an toàn (không bị ghi đè trong cùng lệnh).

### 7.4. Cách 5 — Trường hợp không có cạnh: sắp xếp tăng dần

Không có ràng buộc ⇒ bất đẳng thức sắp xếp lại cho ngay đáp án `Σ sorted(score)[i] · (i+1)`. Dùng để kiểm tra các cách trên ở `n = 22`.

In [10]:
def max_profit_no_edges(score):
    return sum(s * (i + 1) for i, s in enumerate(sorted(score)))

print(max_profit_no_edges([1, 6, 3]), max_profit_dp(3, [], [1, 6, 3]))

25 25


## 8. TRACE từng bước

Trace cách 3 trên DAG 4 đỉnh: `0 → 2`, `1 → 2`, `2 → 3`, `score = [4, 1, 2, 10]`. In các mask hợp lệ, `dp` và chuyển trạng thái.

In [11]:
def trace(n, edges, score):
    pred = [0] * n
    for u, v in edges:
        pred[v] |= 1 << u
    full = (1 << n) - 1
    dp = [-1] * (1 << n)
    dp[0] = 0
    for mask in range(1 << n):
        if dp[mask] < 0:
            continue
        c = bin(mask).count('1') + 1
        moves = []
        for v in range(n):
            if not mask >> v & 1 and pred[v] & mask == pred[v]:
                val = dp[mask] + score[v] * c
                moves.append(f"thêm {v} ở vị trí {c}: {val}")
                if val > dp[mask | 1 << v]:
                    dp[mask | 1 << v] = val
        print(f"mask={format(mask, f'0{n}b')} dp={dp[mask]:>3} | " + ('; '.join(moves) or 'đã đủ'))
    print('Đáp án =', dp[full])
    return dp[full]

trace(4, [[0, 2], [1, 2], [2, 3]], [4, 1, 2, 10])
print('vét cạn:', max_profit_brute(4, [[0, 2], [1, 2], [2, 3]], [4, 1, 2, 10]))

mask=0000 dp=  0 | thêm 0 ở vị trí 1: 4; thêm 1 ở vị trí 1: 1
mask=0001 dp=  4 | thêm 1 ở vị trí 2: 6
mask=0010 dp=  1 | thêm 0 ở vị trí 2: 9
mask=0011 dp=  9 | thêm 2 ở vị trí 3: 15
mask=0111 dp= 15 | thêm 3 ở vị trí 4: 55
mask=1111 dp= 55 | đã đủ
Đáp án = 55
vét cạn: 55


Đọc trace: chỉ có 6 mask hợp lệ (các tập "đóng dưới"); đáp án đặt `1` trước `0` (điểm nhỏ trước) rồi `2`, `3`.

## 9. Code nộp LeetCode

In [12]:
class Solution:
    def maxProfit(self, n: int, edges: List[List[int]], score: List[int]) -> int:
        pred = [0] * n
        for u, v in edges:
            pred[v] |= 1 << u
        if not edges:
            return sum(s * (i + 1) for i, s in enumerate(sorted(score)))
        full = (1 << n) - 1
        size = 1 << n
        dp = [-1] * size
        dp[0] = 0
        pc = [0] * size
        for m in range(1, size):
            pc[m] = pc[m >> 1] + (m & 1)
        for mask in range(size):
            d = dp[mask]
            if d < 0:
                continue
            c = pc[mask] + 1
            rest = full ^ mask
            while rest:
                lb = rest & -rest
                rest ^= lb
                v = lb.bit_length() - 1
                p = pred[v]
                if p & mask == p:
                    val = d + score[v] * c
                    nm = mask | lb
                    if val > dp[nm]:
                        dp[nm] = val
        return dp[full]

for n, e, s, exp in examples:
    got = Solution().maxProfit(n, e, s)
    print(got, 'OK' if got == exp else 'SAI')

8 OK
25 OK


## 10. Vì sao thuật toán đúng

1. **Tách lợi nhuận theo tiền tố:** trong một thứ tự `v_1, …, v_n`, đỉnh `v_{c+1}` đóng góp `score · (c+1)`, với `c` = số đỉnh đứng trước — chỉ phụ thuộc **kích thước** tập tiền tố.
2. **Tập tiền tố hợp lệ** là tập đóng dưới (chứa mọi tiền nhiệm của các phần tử của nó). Một tiền tố mở rộng hợp lệ bằng `v` ⇔ `v` có mọi tiền nhiệm trong tập.
3. **Tối ưu cấu trúc con:** lợi nhuận tốt nhất cho tập `S ∪ {v}` mà `v` đứng cuối = lợi nhuận tốt nhất cho `S` + `score[v]·(|S|+1)`; lấy max trên mọi `v` có thể đứng cuối. DP đẩy (push) ở trên tương đương.
4. **Thứ tự duyệt:** `mask | 1<<v > mask` nên duyệt mask tăng dần đảm bảo `dp[mask]` đã cuối cùng khi được dùng; lớp popcount trong numpy cũng đúng thứ tự phụ thuộc.
5. **Không cạnh:** bất đẳng thức sắp xếp lại ⇒ sắp tăng dần tối ưu.

## 11. Phân tích Time / Space Complexity

Gọi `n ≤ 22`, `m` số cạnh.

| Cách | Time | Space |
|---|---|---|
| 1. Hoán vị | `O(n! (n + m))` | `O(n)` |
| 2. Đệ quy có nhớ | `O(2^n · n)` (+ chi phí gọi hàm) | `O(2^n)` |
| 3. Bottom-up Python | `O(2^n · n)` | `O(2^n)` |
| 4. Numpy theo lớp | `O(2^n · n)` phép toán mảng | `O(2^n)` |
| 5. Không cạnh | `O(n log n)` | `O(n)` |

Số mask thực sự hợp lệ = số tập đóng dưới của DAG; càng nhiều cạnh càng ít.

## 12. Bộ test

### 12.1. Ví dụ đề + ca biên

In [13]:
impls = {
    'memo': max_profit_memo,
    'dp': max_profit_dp,
    'numpy': max_profit_numpy,
    'submit': lambda n, e, s: Solution().maxProfit(n, e, s),
}
edge = [
    (1, [], [7], 7),
    (3, [], [3, 1, 2], 1 * 1 + 2 * 2 + 3 * 3),
    (3, [[0, 1], [1, 2]], [9, 5, 1], 9 + 10 + 3),              # chuỗi: thứ tự bắt buộc
    (4, [[0, 1], [2, 3]], [10, 1, 1, 10], None),               # kiểm bằng vét cạn
]
for n, e, s, exp in examples + edge:
    b = max_profit_brute(n, e, s)
    if exp is not None:
        assert b == exp
    for name, f in impls.items():
        assert f(n, e, s) == b, (name, n, e, s)
print('Ví dụ đề và ca biên: tất cả đúng.')

Ví dụ đề và ca biên: tất cả đúng.


### 12.2. Đối chiếu ngẫu nhiên với vét cạn (DAG ngẫu nhiên)

In [14]:
import random
random.seed(3530)

def rand_dag(n, p):
    perm = list(range(n))
    random.shuffle(perm)                 # thứ tự topo ẩn
    es = []
    for i in range(n):
        for j in range(i + 1, n):
            if random.random() < p:
                es.append([perm[i], perm[j]])
    random.shuffle(es)
    return es

cnt = 0
for _ in range(300):
    n = random.randint(1, 7)
    e = rand_dag(n, random.choice([0.0, 0.2, 0.5, 0.9]))
    s = [random.randint(1, 20) for _ in range(n)]
    exp = max_profit_brute(n, e, s)
    for name, f in impls.items():
        assert f(n, e, s) == exp, (name, n, e, s)
    cnt += 1
print('Khớp vét cạn trên', cnt, 'DAG ngẫu nhiên.')

Khớp vét cạn trên 300 DAG ngẫu nhiên.


### 12.3. Ca lớn (`n = 20` và `n = 22`)

In [15]:
import time
random.seed(22)
n = 20
for p in [0.0, 0.05, 0.3]:
    e = rand_dag(n, p)
    s = [random.randint(1, 10**5) for _ in range(n)]
    t = time.time(); a = max_profit_dp(n, e, s); t1 = time.time() - t
    t = time.time(); b = max_profit_numpy(n, e, s); t2 = time.time() - t
    assert a == b
    print(f"n=20, mật độ cạnh {p}: {a} | Python {t1:.2f}s | numpy {t2:.2f}s")
n = 22
e = [[0, 1]]                               # chỉ một cạnh: gần như mọi mask đều hợp lệ (trường hợp xấu)
s = [random.randint(1, 10**5) for _ in range(n)]
t = time.time(); b = max_profit_numpy(n, e, s); t2 = time.time() - t
t = time.time(); a = max_profit_dp(n, e, s); t1 = time.time() - t
assert a == b
print(f"n=22, {len(e)} cạnh: Python thuần {t1:.2f}s | numpy {t2:.2f}s")
s2 = [random.randint(1, 10**5) for _ in range(22)]
assert max_profit_numpy(22, [], s2) == max_profit_no_edges(s2)
print('n=22 không cạnh: numpy khớp công thức sắp xếp.')

n=20, mật độ cạnh 0.0: 15181967 | Python 3.09s | numpy 0.26s


n=20, mật độ cạnh 0.05: 13191972 | Python 0.17s | numpy 0.09s
n=20, mật độ cạnh 0.3: 12442288 | Python 0.09s | numpy 0.08s


n=22, 1 cạnh: Python thuần 9.69s | numpy 1.06s


n=22 không cạnh: numpy khớp công thức sắp xếp.


**Ghi chú hiệu năng:** ở giới hạn `n = 22` với rất ít cạnh, lời giải Python thuần mất cỡ chục giây (số tập đóng dưới gần `2^22`); bản numpy chỉ khoảng 1–2 giây. Khi đồ thị có nhiều cạnh, số mask hợp lệ giảm mạnh và Python thuần chạy nhanh. Trường hợp không cạnh được xử lý riêng bằng sắp xếp.

## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Tham lam: luôn chọn đỉnh sẵn sàng có điểm **nhỏ nhất** trước

In [16]:
import heapq

def greedy_min_first(n, edges, score):
    indeg = [0] * n
    adj = [[] for _ in range(n)]
    for u, v in edges:
        adj[u].append(v); indeg[v] += 1
    pq = [(score[v], v) for v in range(n) if indeg[v] == 0]
    heapq.heapify(pq)
    order = []
    while pq:
        _, u = heapq.heappop(pq)
        order.append(u)
        for v in adj[u]:
            indeg[v] -= 1
            if indeg[v] == 0:
                heapq.heappush(pq, (score[v], v))
    return profit(order, score)

random.seed(1)
found = None
for _ in range(3000):
    n = random.randint(3, 6)
    e = rand_dag(n, 0.3)
    s = [random.randint(1, 20) for _ in range(n)]
    if greedy_min_first(n, e, s) != max_profit_brute(n, e, s):
        found = (n, e, s); break
print('phản ví dụ:', found, '| tham lam:', greedy_min_first(*found), '| đúng:', Solution().maxProfit(*found))

phản ví dụ: (3, [[2, 0]], [4, 11, 17]) | tham lam: 57 | đúng: 58


**Vì sao sai:** đỉnh điểm nhỏ có thể đang "khóa" một đỉnh điểm rất lớn; xử lý nó sớm hay muộn ảnh hưởng tới cả chuỗi phía sau. Tham lam cục bộ không thấy được điều đó.

### Lỗi 2 — Quên kiểm tra tiền nhiệm (`pred[v] & mask == pred[v]`)

In [17]:
def wrong_no_pred(n, edges, score):
    return max_profit_dp(n, [], score)          # SAI: bỏ qua ràng buộc

case = (2, [[1, 0]], [2, 3])            # bắt buộc 1 trước 0
print('sai :', wrong_no_pred(*case), '| đúng:', Solution().maxProfit(*case))

sai : 8 | đúng: 7


**Vì sao sai:** đỉnh `1` (điểm 3) bắt buộc đứng trước `0` (điểm 2); bỏ ràng buộc cho phép đặt điểm lớn ở cuối tùy ý ⇒ tính dư.

### Lỗi 3 — Kiểm tra tiền nhiệm bằng `pred[v] & mask != 0` (chỉ cần **một** tiền nhiệm)

In [18]:
def wrong_any_pred(n, edges, score):
    pred = [0] * n
    for u, v in edges:
        pred[v] |= 1 << u
    full = (1 << n) - 1
    dp = [-1] * (1 << n); dp[0] = 0
    for mask in range(1 << n):
        if dp[mask] < 0:
            continue
        c = bin(mask).count('1') + 1
        for v in range(n):
            if not mask >> v & 1 and (pred[v] == 0 or pred[v] & mask):     # SAI
                dp[mask | 1 << v] = max(dp[mask | 1 << v], dp[mask] + score[v] * c)
    return dp[full]

random.seed(9)
case = None
for _ in range(5000):
    n = random.randint(3, 6)
    e = rand_dag(n, 0.5)
    sc = [random.randint(1, 20) for _ in range(n)]
    if wrong_any_pred(n, e, sc) != max_profit_brute(n, e, sc):
        case = (n, e, sc); break
print('phản ví dụ:', case)
print('sai :', wrong_any_pred(*case), '| đúng:', Solution().maxProfit(*case))

phản ví dụ: (6, [[3, 2], [0, 2], [2, 4], [1, 0], [3, 4], [5, 2], [5, 4], [1, 4]], [8, 14, 3, 9, 7, 13])
sai : 210 | đúng: 170


**Vì sao sai:** `v` cần **mọi** tiền nhiệm đã được xử lý, không chỉ một.

### Lỗi 4 — Vị trí đánh số từ 0

In [19]:
def wrong_zero_based(order, score):
    return sum(score[v] * i for i, v in enumerate(order))

print('sai :', wrong_zero_based([0, 2, 1], [1, 6, 3]), '| đúng:', profit([0, 2, 1], [1, 6, 3]))

sai : 15 | đúng: 25


**Vì sao sai:** đề dùng vị trí **1-based**.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Chọn thứ tự cho `n ≤ 20–22` phần tử, chi phí phụ thuộc vị trí" ⇒ **bitmask DP trên tập đã xếp**: vị trí tiếp theo = popcount + 1.
- "Ràng buộc thứ tự (DAG)" ⇒ bitmask tiền nhiệm; tập hợp lệ là tập đóng dưới.
- Python với `2^22` trạng thái ⇒ cân nhắc vector hóa (numpy) hoặc cắt các mask không hợp lệ sớm (`dp < 0`).
- Không ràng buộc ⇒ bất đẳng thức sắp xếp lại.

## 15. Biến thể của bài

1. **Tối thiểu** lợi nhuận ⇒ đổi `max` thành `min` (không ràng buộc: sắp giảm dần).
2. Đếm số thứ tự topo đạt lợi nhuận lớn nhất ⇒ DP thêm bộ đếm.
3. Đếm số thứ tự topo (không trọng số) ⇒ `cnt[mask]` cộng dồn — bài kinh điển.
4. Bài liên quan: LC 1494 *Parallel Courses II*, LC 2741 *Special Permutations*, lập lịch `1|prec|ΣwC` (NP-khó tổng quát).

In [20]:
# Biến thể 3: đếm số thứ tự topo
def count_topo(n, edges):
    pred = [0] * n
    for u, v in edges:
        pred[v] |= 1 << u
    cnt = [0] * (1 << n); cnt[0] = 1
    for mask in range(1 << n):
        if cnt[mask]:
            for v in range(n):
                if not mask >> v & 1 and pred[v] & mask == pred[v]:
                    cnt[mask | 1 << v] += cnt[mask]
    return cnt[-1]

e = [[0, 2], [1, 2], [2, 3]]
print(count_topo(4, e), sum(1 for p in permutations(range(4)) if is_topological(p, e)))

2 2


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1 | Hoán vị | `O(n!)` | `O(n)` | Oracle |
| 2 | Đệ quy có nhớ | `O(2^n n)` | `O(2^n)` | Dễ viết, `n` vừa |
| 3 | Bottom-up Python | `O(2^n n)` | `O(2^n)` | **Nộp bài** (thuần Python) |
| 4 | Numpy theo lớp | `O(2^n n)` (C) | `O(2^n)` | Khi được dùng numpy, `n = 22` |
| 5 | Sắp xếp | `O(n log n)` | `O(n)` | Không có cạnh |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `maxProfit(3, [], [2, 2, 2])`.

**Bài 2 (tính tay):** `maxProfit(3, [[2, 0]], [5, 1, 3])`.

**Bài 3 (điền TODO):** viết `best_order(n, edges, score)` trả về **thứ tự** tối ưu.

**Bài 4 (tự viết lại):** biến thể 1 — lợi nhuận **nhỏ nhất**.

**Bài 5 (mini challenge):** biến thể 2 — đếm số thứ tự topo đạt lợi nhuận lớn nhất.

In [21]:
def best_order_todo(n, edges, score):
    # TODO: lưu parent[mask] khi cập nhật dp
    raise NotImplementedError

def min_profit_todo(n, edges, score):
    # TODO
    raise NotImplementedError

def count_optimal_todo(n, edges, score):
    # TODO
    raise NotImplementedError

print('Đã định nghĩa các hàm TODO (không gọi).')

Đã định nghĩa các hàm TODO (không gọi).


## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Mọi thứ tự như nhau. (2) `2·(1+2+3)`. (3) `12`.

**Bài 2.** (1) `2` phải trước `0`. (2) Thử `1,2,0`: `1 + 6 + 15 = 22`; `2,0,1`: `3 + 10 + 3 = 16`; `2,1,0`: `3 + 2 + 15 = 20`. (3) `22`.

**Bài 3.** (1) Khi `dp[nm]` được cải thiện, lưu `parent[nm] = (mask, v)`. (2) Lần ngược từ `full`. (3) Kiểm tra hợp lệ và tổng.

**Bài 4.** (1) Khởi tạo `dp = +∞`, dùng `min`. (2) Không ràng buộc: sắp giảm dần. (3) Vét cạn kiểm tra.

**Bài 5.** (1) Lưu `(dp, cnt)`. (2) Bằng giá trị thì cộng `cnt`, lớn hơn thì thay. (3) Vét cạn đếm hoán vị đạt max.

## 19. Đáp án tham khảo cho bài tập

In [22]:
# Bài 1, 2
a1 = Solution().maxProfit(3, [], [2, 2, 2])
a2 = Solution().maxProfit(3, [[2, 0]], [5, 1, 3])
assert a1 == 12 and a2 == 22 == max_profit_brute(3, [[2, 0]], [5, 1, 3])
print('Bài 1:', a1, '| Bài 2:', a2)

Bài 1: 12 | Bài 2: 22


In [23]:
# Bài 3, 4
def best_order(n, edges, score):
    pred = [0] * n
    for u, v in edges:
        pred[v] |= 1 << u
    full = (1 << n) - 1
    dp = [-1] * (1 << n); dp[0] = 0
    parent = [None] * (1 << n)
    for mask in range(1 << n):
        if dp[mask] < 0:
            continue
        c = bin(mask).count('1') + 1
        for v in range(n):
            if not mask >> v & 1 and pred[v] & mask == pred[v]:
                val = dp[mask] + score[v] * c
                if val > dp[mask | 1 << v]:
                    dp[mask | 1 << v] = val
                    parent[mask | 1 << v] = (mask, v)
    order, m = [], full
    while m:
        m, v = parent[m]
        order.append(v)
    return order[::-1]

def min_profit(n, edges, score):
    pred = [0] * n
    for u, v in edges:
        pred[v] |= 1 << u
    INF = float('inf')
    dp = [INF] * (1 << n); dp[0] = 0
    for mask in range(1 << n):
        if dp[mask] == INF:
            continue
        c = bin(mask).count('1') + 1
        for v in range(n):
            if not mask >> v & 1 and pred[v] & mask == pred[v]:
                dp[mask | 1 << v] = min(dp[mask | 1 << v], dp[mask] + score[v] * c)
    return dp[-1]

random.seed(3)
for _ in range(200):
    n = random.randint(1, 7)
    e = rand_dag(n, 0.3)
    s = [random.randint(1, 20) for _ in range(n)]
    o = best_order(n, e, s)
    assert sorted(o) == list(range(n)) and is_topological(o, e) and profit(o, s) == max_profit_brute(n, e, s)
    assert min_profit(n, e, s) == min(profit(p, s) for p in permutations(range(n)) if is_topological(p, e))
print('Bài 3: ví dụ', best_order(4, [[0, 2], [1, 2], [2, 3]], [4, 1, 2, 10]), '| Bài 4: min đúng')

Bài 3: ví dụ [1, 0, 2, 3] | Bài 4: min đúng


In [24]:
# Bài 5: đếm số thứ tự tối ưu
def count_optimal(n, edges, score):
    pred = [0] * n
    for u, v in edges:
        pred[v] |= 1 << u
    dp = [-1] * (1 << n); cnt = [0] * (1 << n)
    dp[0], cnt[0] = 0, 1
    for mask in range(1 << n):
        if dp[mask] < 0:
            continue
        c = bin(mask).count('1') + 1
        for v in range(n):
            if not mask >> v & 1 and pred[v] & mask == pred[v]:
                nm, val = mask | 1 << v, dp[mask] + score[v] * c
                if val > dp[nm]:
                    dp[nm], cnt[nm] = val, cnt[mask]
                elif val == dp[nm]:
                    cnt[nm] += cnt[mask]
    return dp[-1], cnt[-1]

random.seed(4)
for _ in range(200):
    n = random.randint(1, 6)
    e = rand_dag(n, 0.3)
    s = [random.randint(1, 4) for _ in range(n)]
    vals = [profit(p, s) for p in permutations(range(n)) if is_topological(p, e)]
    assert count_optimal(n, e, s) == (max(vals), vals.count(max(vals)))
print('Bài 5: ví dụ', count_optimal(3, [], [2, 2, 2]))

Bài 5: ví dụ (12, 6)


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
pred[v] = bitmask tiền nhiệm
dp = [-1] * 2^n; dp[0] = 0
for mask tăng dần (bỏ dp < 0):
    c = popcount(mask) + 1
    for v ∉ mask với pred[v] ⊆ mask:
        dp[mask | 1<<v] = max(..., dp[mask] + score[v] * c)
return dp[full]
```

**Checklist:**

- [ ] Vị trí 1-based = popcount + 1?
- [ ] Kiểm tra **mọi** tiền nhiệm (`pred & mask == pred`)?
- [ ] Bỏ qua mask không hợp lệ (`dp < 0`)?
- [ ] Trường hợp không cạnh → sắp xếp (nhanh)?
- [ ] Ước lượng thời gian `2^22` trong Python (cân nhắc numpy)?